# 6.15 · 自编码器 / Autoencoder

> **课程定位 / Where this fits**
> PCA(6.8)是**线性**降维。自编码器(Autoencoder)用神经网络做**非线性**降维: 一个 **encoder** 把输入压到低维**瓶颈(bottleneck)**, 一个 **decoder** 从瓶颈重构输入, 训练目标是让重构尽量接近原图。它是 PCA 的非线性深度版, 也是表示学习、去噪、异常检测、以及生成模型 **VAE** 的前身。本课用 PyTorch 搭一个小自编码器。
> An autoencoder is a neural network that compresses input to a low-dim bottleneck (encoder) and reconstructs it (decoder) — nonlinear PCA, and the precursor to VAEs.

> 💡 **面试相关 / Interview-relevant**
> - "自编码器结构 / 瓶颈层的作用" ★★★★★
> - "线性自编码器 + MSE 与 PCA 的关系" ★★★★★（等价于 PCA 子空间）
> - "为什么非线性激活让 AE 超越 PCA" ★★★★
> - "欠完备 vs 过完备; 去噪自编码器" ★★★★
> - "AE vs VAE 区别" ★★★（VAE 是概率生成式）

---

## 学习目标 / Learning Objectives
1. encoder–bottleneck–decoder 结构 + 重构损失。
2. PyTorch 实现 + 训练。
3. 非线性 AE vs 线性 AE/PCA。
4. 潜空间可视化 + 去噪自编码器。

## 目录 / TOC
1. [结构与 PCA 的关系 ⭐](#1)
2. [👕 数据: Digits(Fashion-MNIST 替身)](#2)
3. [PyTorch 自编码器 ⭐](#3)
4. [重构 + 潜空间 + vs PCA ⭐](#4)
5. [去噪自编码器](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 结构与 PCA 的关系 ⭐ / Architecture & Link to PCA

**结构**: $\mathbf{x}\xrightarrow{\text{encoder } f}\mathbf{z}\xrightarrow{\text{decoder } g}\hat{\mathbf{x}}$, 其中 $\mathbf{z}$ 是低维**瓶颈/潜表示(latent code)**。**欠完备(undercomplete)**自编码器的瓶颈维度 < 输入维度, 迫使网络学会压缩。训练最小化**重构损失**(连续值用 MSE):
$$\mathcal{L} = \frac1n\sum_i\|\mathbf{x}_i - g(f(\mathbf{x}_i))\|^2$$

**与 PCA 的关系**(面试核心): 若 encoder/decoder 都是**线性**且损失是 MSE, 自编码器学到的瓶颈子空间**等价于 PCA**(张成同一个主子空间)。**加上非线性激活(ReLU 等), AE 就能学弯曲流形**, 超越 PCA 的线性局限——这正是它的价值。


<a id="2"></a>
## 2. 数据: Digits(Fashion-MNIST 替身) / Digits as a Stand-in

README 里规划的是 Fashion-MNIST, 但它需要联网下载约 30MB。为保持 notebook **自包含、秒级可跑**, 这里用 sklearn 内置 **Digits**(8×8=64 维手写数字, 像素已在 0–16, 归一化到 [0,1])作替身——结构(图像像素重构)与 Fashion-MNIST 完全一致, 换成真数据只需替换加载这一步。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")
torch.manual_seed(0); np.random.seed(0)
print("torch", torch.__version__)

digits = load_digits()
X = digits.data / 16.0           # 归一化到 [0,1]
y = digits.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
Xtr_t = torch.tensor(X_tr, dtype=torch.float32)
Xte_t = torch.tensor(X_te, dtype=torch.float32)
print(f"Digits: {X.shape} → 64 维像素, 训练 {len(X_tr)} / 测试 {len(X_te)}")


<a id="3"></a>
## 3. PyTorch 自编码器 ⭐ / The Autoencoder in PyTorch

64 → 32 → **2(瓶颈)** → 32 → 64, 中间用 ReLU 非线性, 输出 Sigmoid(像素在 [0,1])。瓶颈设 2 维便于可视化潜空间。


In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, d_in=64, d_latent=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, 32), nn.ReLU(),
            nn.Linear(32, d_latent))                 # 瓶颈
        self.decoder = nn.Sequential(
            nn.Linear(d_latent, 32), nn.ReLU(),
            nn.Linear(32, d_in), nn.Sigmoid())
    def forward(self, x):
        z = self.encoder(x); return self.decoder(z), z

model = Autoencoder(64, 2)
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

losses = []
for epoch in range(300):
    opt.zero_grad()
    recon, _ = model(Xtr_t)
    loss = loss_fn(recon, Xtr_t)
    loss.backward(); opt.step()
    losses.append(loss.item())
print(f"训练完成. 最终训练重构 MSE: {losses[-1]:.4f}")
with torch.no_grad():
    test_mse = loss_fn(model(Xte_t)[0], Xte_t).item()
print(f"测试重构 MSE: {test_mse:.4f}")

fig, ax = plt.subplots(figsize=(7,3.5))
ax.plot(losses); ax.set_xlabel("epoch"); ax.set_ylabel("重构 MSE")
ax.set_title("自编码器训练曲线(重构损失下降)")
plt.tight_layout(); plt.show()


<a id="4"></a>
## 4. 重构 + 潜空间 + vs PCA ⭐ / Reconstruction, Latent, vs PCA


In [ ]:
with torch.no_grad():
    recon_te, Z_ae = model(Xte_t)
recon_te = recon_te.numpy(); Z_ae = Z_ae.numpy()

# 重构对比 / reconstructions
fig, axes = plt.subplots(2, 8, figsize=(11, 3))
for j in range(8):
    axes[0,j].imshow(X_te[j].reshape(8,8), cmap="gray_r"); axes[0,j].axis("off")
    axes[1,j].imshow(recon_te[j].reshape(8,8), cmap="gray_r"); axes[1,j].axis("off")
fig.text(0.08, 0.72, "原始", va="center"); fig.text(0.08, 0.3, "AE重构", va="center")
plt.suptitle("自编码器重构(经过 2 维瓶颈, 仍认得出数字)"); plt.tight_layout(); plt.show()

# 潜空间 vs PCA / 2D latent vs PCA
from sklearn.decomposition import PCA
Z_pca = PCA(2).fit(X_tr).transform(X_te)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
s0=axes[0].scatter(Z_ae[:,0], Z_ae[:,1], c=y_te, cmap="tab10", s=12); axes[0].set_title("AE 2D 潜空间(非线性)")
axes[1].scatter(Z_pca[:,0], Z_pca[:,1], c=y_te, cmap="tab10", s=12); axes[1].set_title("PCA 2D(线性)")
plt.colorbar(s0, ax=axes[1], label="digit"); plt.tight_layout(); plt.show()
print("AE 用非线性瓶颈学表示; 线性AE+MSE 会退化成 PCA 子空间, 加 ReLU 才能弯曲")


<a id="5"></a>
## 5. 去噪自编码器 / Denoising Autoencoder

**去噪自编码器(DAE)**: 训练时给输入**加噪**, 但要求重构出**干净**原图。这逼网络学到更鲁棒的特征(不能只是恒等复制), 是自监督表示学习的经典技巧。


In [ ]:
dae = Autoencoder(64, 8)         # 瓶颈大一点以便重构更清晰
opt = torch.optim.Adam(dae.parameters(), lr=1e-2)
for epoch in range(400):
    opt.zero_grad()
    noisy = (Xtr_t + 0.3*torch.randn_like(Xtr_t)).clamp(0,1)   # 加噪输入
    recon, _ = dae(noisy)
    loss = loss_fn(recon, Xtr_t)                                # 目标=干净图
    loss.backward(); opt.step()

with torch.no_grad():
    noisy_te = (Xte_t + 0.3*torch.randn_like(Xte_t)).clamp(0,1)
    denoised = dae(noisy_te)[0].numpy()
fig, axes = plt.subplots(3, 8, figsize=(11, 4.2))
for j in range(8):
    axes[0,j].imshow(X_te[j].reshape(8,8), cmap="gray_r"); axes[0,j].axis("off")
    axes[1,j].imshow(noisy_te[j].numpy().reshape(8,8), cmap="gray_r"); axes[1,j].axis("off")
    axes[2,j].imshow(denoised[j].reshape(8,8), cmap="gray_r"); axes[2,j].axis("off")
for yv,t in zip([0.78,0.5,0.22], ["原始","加噪","去噪"]): fig.text(0.08, yv, t, va="center")
plt.suptitle("去噪自编码器: 输入加噪, 训练重构干净图 → 学到鲁棒特征"); plt.tight_layout(); plt.show()
print("DAE 不能简单恒等复制(输入是脏的), 被迫学到数据的本质结构 → 自监督表示学习思想")


<a id="6"></a>
## 6. 小结 / Summary

```
自编码器: encoder(x→z 瓶颈) + decoder(z→x̂); 最小化重构损失(MSE)
欠完备(瓶颈<输入)迫使学压缩表示; 线性AE+MSE ≡ PCA 子空间, 加非线性激活才超越 PCA
潜空间 z 是非线性低维表示, 可可视化/做下游特征
去噪自编码器(DAE): 输入加噪→重构干净, 学鲁棒特征(自监督)
是 VAE(概率生成)、表示学习、异常检测(高重构误差=异常)的基础
```

### 💡 面试速查
1. **结构**: encoder→瓶颈 z→decoder; 重构损失 MSE
2. **线性 AE + MSE = PCA**(同一子空间); **非线性激活**让它学弯曲流形超越 PCA
3. **欠完备**瓶颈迫使压缩; 过完备需正则(稀疏/去噪)防恒等复制
4. **去噪 AE**: 加噪输入→干净输出, 自监督学鲁棒特征
5. **AE vs VAE**: VAE 是概率生成式(潜变量有分布), 能采样生成新样本

### 下一节
**6.16 异常检测**——找"不正常"的点。One-Class SVM、Isolation Forest、LOF, 以及用重构误差(本课 AE)做异常检测。
